# Stage 04a — Model Building (MIV Stepwise Selection)

**Purpose:** Build a logistic regression PD model using Marginal Information Value (MIV) stepwise variable selection from the approved shortlist.

**Inputs:**
- Binned dataset: `{RUN_DIR}/data/loans_binned.csv` (bin labels from Stage 03 OptimalBinning)
- Shortlist: 8 variables from Stage 03 (IV >= 0.10)
- Target: `Creditability` (1 = default)

**Method:** MIV stepwise logistic regression with WoE coding

In [ ]:
import sys, os

# Ensure working directory is project root
os.chdir('C:/projects/superagent')
sys.path.insert(0, 'src')

import pdtoolkit as pdt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

RUN_DIR = 'runs/2026-03-17_071354'

# Plot config
BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

## 1. Load Data and Shortlist

In [ ]:
# Load binned dataset (bin labels are strings from OptimalBinning)
db_binned = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')
print(f'Binned dataset shape: {db_binned.shape}')
print(f'Default rate: {db_binned["Creditability"].mean():.4f}')

# Shortlisted variables from Stage 03
shortlist = [
    'Account Balance',
    'Payment Status of Previous Credit',
    'Duration of Credit (month)',
    'Value Savings/Stocks',
    'Purpose',
    'Credit Amount',
    'Most valuable available asset',
    'Age (years)'
]

target = 'Creditability'

# Subset to shortlist + target
db_model = db_binned[shortlist + [target]].copy()

# Ensure all shortlist columns are string type (required by step_miv/replace_woe)
for col in shortlist:
    db_model[col] = db_model[col].astype(str)

# Ensure target is numeric 0/1
db_model[target] = db_model[target].astype(float).astype(int)

print(f'Model dataset shape: {db_model.shape}')
print(f'\nUnique bins per variable:')
for col in shortlist:
    print(f'  {col}: {db_model[col].nunique()} bins')

## 2. WoE Encoding and Mapping Extraction

In [ ]:
# Use pdt.replace_woe to get the WoE-encoded dataset and mappings
db_woe, woe_info = pdt.replace_woe(db_model, target=target)

# Also compute bivariate analysis to get per-variable WoE tables for mapping storage
biv_results, biv_info = pdt.bivariate(db_model, target=target)

# Extract WoE mappings per variable
woe_mappings = {}
for var in shortlist:
    var_biv = biv_results[biv_results['rf'] == var][['bin', 'woe']].copy()
    woe_mappings[var] = [{'bin': row['bin'], 'woe': round(float(row['woe']), 6)} for _, row in var_biv.iterrows()]

print('WoE mappings extracted for all shortlisted variables')
for var in shortlist:
    print(f'  {var}: {len(woe_mappings[var])} bins')
    for m in woe_mappings[var]:
        print(f'    {m["bin"]}: WoE = {m["woe"]:.4f}')

## 3. MIV Stepwise Variable Selection

In [ ]:
# Run MIV stepwise selection with default thresholds
miv_threshold = 0.02
m_ch_p_val = 0.05

miv_result = pdt.step_miv(
    start_model=f'{target} ~ 1',
    miv_threshold=miv_threshold,
    m_ch_p_val=m_ch_p_val,
    coding='WoE',
    db=db_model
)

# Extract selected variables
if len(miv_result.steps) > 0:
    selected_variables = miv_result.steps['rf_miv'].tolist()
else:
    selected_variables = []

excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]

print(f'MIV threshold: {miv_threshold}')
print(f'Variables selected: {len(selected_variables)}')
print(f'Selected: {selected_variables}')
print(f'Excluded from shortlist: {excluded_from_shortlist}')

# Print step details
print('\nStepwise iterations:')
print(miv_result.steps[['rf_miv', 'miv', 'p_val']].to_string(index=False))

# Print warnings if any
if len(miv_result.warnings) > 0:
    print('\nWarnings:')
    print(miv_result.warnings.to_string(index=False))

In [ ]:
# Self-assessment check 1: Variable count
threshold_adjusted = False
if len(selected_variables) < 4:
    print(f'WARNING: Only {len(selected_variables)} variables selected. Relaxing threshold by 50%.')
    miv_threshold_new = miv_threshold * 0.5
    miv_result = pdt.step_miv(
        start_model=f'{target} ~ 1',
        miv_threshold=miv_threshold_new,
        m_ch_p_val=m_ch_p_val,
        coding='WoE',
        db=db_model
    )
    if len(miv_result.steps) > 0:
        selected_variables = miv_result.steps['rf_miv'].tolist()
    excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]
    miv_threshold = miv_threshold_new
    threshold_adjusted = True
    print(f'After relaxation: {len(selected_variables)} variables selected')
elif len(selected_variables) > 12:
    print(f'WARNING: {len(selected_variables)} variables selected. Tightening threshold by 50%.')
    miv_threshold_new = miv_threshold * 1.5
    miv_result = pdt.step_miv(
        start_model=f'{target} ~ 1',
        miv_threshold=miv_threshold_new,
        m_ch_p_val=m_ch_p_val,
        coding='WoE',
        db=db_model
    )
    if len(miv_result.steps) > 0:
        selected_variables = miv_result.steps['rf_miv'].tolist()
    excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]
    miv_threshold = miv_threshold_new
    threshold_adjusted = True
    print(f'After tightening: {len(selected_variables)} variables selected')
else:
    print(f'Variable count check PASS: {len(selected_variables)} variables selected (4-12 range)')

## 4. Refit Final Model with Statsmodels

In [ ]:
# Get the WoE-encoded development database from step_miv
dev_db = miv_result.dev_db.copy()

# The dev_db has WoE-encoded selected variables (step_miv replaces them)
X_woe = dev_db[selected_variables].copy()
y = dev_db[target].copy()

# Verify WoE columns are numeric
print('WoE column types:')
for col in selected_variables:
    print(f'  {col}: dtype={X_woe[col].dtype}, sample values: {X_woe[col].unique()[:5]}')

# Refit with statsmodels
X = sm.add_constant(X_woe)
logit_model = sm.Logit(y, X)
logit_res = logit_model.fit()

print('\n' + '='*70)
print(logit_res.summary())
print('='*70)

In [ ]:
# Extract coefficient table
summary_df = pd.DataFrame({
    'variable': ['const'] + selected_variables,
    'coef': logit_res.params.values,
    'std_err': logit_res.bse.values,
    'z': logit_res.tvalues.values,
    'p_value': logit_res.pvalues.values,
    'ci_lower': logit_res.conf_int()[0].values,
    'ci_upper': logit_res.conf_int()[1].values
})

print('Coefficient Table:')
print(summary_df.to_string(index=False))

# Extract model-level statistics
model_stats = {
    'nobs': int(logit_res.nobs),
    'df_model': int(logit_res.df_model),
    'pseudo_r2': float(logit_res.prsquared),
    'llf': float(logit_res.llf),
    'llnull': float(logit_res.llnull),
    'llr_pvalue': float(logit_res.llr_pvalue),
    'converged': bool(logit_res.mle_retvals['converged'])
}

print(f'\nModel Statistics:')
for k, v in model_stats.items():
    print(f'  {k}: {v}')

## 5. Self-Assessment Checks

In [ ]:
# Compute predicted probabilities
y_pred_prob = logit_res.predict(X)

# ---- Check 2: Coefficient sign consistency ----
# pdtoolkit WoE = ln(dist_good/dist_bad): positive WoE = lower risk
# With target Creditability=1 (default), logit models P(default=1)
# Therefore coefficients on WoE should be NEGATIVE:
#   higher WoE (lower risk) => lower P(default) => negative coefficient
sign_results = []
all_signs_consistent = True
for var in selected_variables:
    coef = logit_res.params[var]
    consistent = coef < 0  # WoE coefficients should be negative when target=1 is default
    if not consistent:
        all_signs_consistent = False
    sign_results.append({
        'variable': var,
        'coefficient': round(float(coef), 6),
        'woe_direction_consistent': consistent
    })

print('Check 2 - Coefficient Sign Consistency:')
print('  (WoE = ln(dist_good/dist_bad), target=1 is default => coefficients should be negative)')
for r in sign_results:
    status = 'PASS' if r['woe_direction_consistent'] else 'FAIL'
    print(f'  {r["variable"]}: coef={r["coefficient"]:.4f} [{status}]')
print(f'  Overall: {"PASS" if all_signs_consistent else "FAIL"}')

# ---- Check 3: VIF ----
vif_data = []
vif_flag = False
for i, var in enumerate(selected_variables):
    vif_val = variance_inflation_factor(X_woe.values, i)
    if vif_val > 5:
        vif_flag = True
    vif_data.append({'variable': var, 'vif': round(float(vif_val), 4)})

print(f'\nCheck 3 - VIF:')
for v in vif_data:
    status = 'WARN' if v['vif'] > 5 else 'PASS'
    print(f'  {v["variable"]}: VIF={v["vif"]:.2f} [{status}]')

# Merge sign and VIF results
for sr in sign_results:
    for vd in vif_data:
        if sr['variable'] == vd['variable']:
            sr['vif'] = vd['vif']

In [ ]:
# ---- Model Performance Metrics ----
model_auc = pdt.auc_model(y_pred_prob.values, y.values)
model_gini = 2 * model_auc - 1

# KS statistic
fpr, tpr, thresholds = roc_curve(y, y_pred_prob)
model_ks = float(np.max(tpr - fpr))

print(f'Model Performance:')
print(f'  AUC:  {model_auc:.4f}')
print(f'  Gini: {model_gini:.4f}')
print(f'  KS:   {model_ks:.4f}')

# AUC benchmark assessment
if model_auc < 0.60:
    auc_assessment = 'Unacceptable'
elif model_auc < 0.70:
    auc_assessment = 'Weak'
elif model_auc < 0.80:
    auc_assessment = 'Acceptable'
else:
    auc_assessment = 'Strong'
print(f'  AUC Assessment: {auc_assessment}')

In [ ]:
# ---- Check 4: Score Distribution ----
scores = pdt.scaled_score(y_pred_prob.values, score=600, odd=50, pdo=20)

score_min = float(np.min(scores))
score_max = float(np.max(scores))
score_mean = float(np.mean(scores))
score_std = float(np.std(scores))
pct_below_400 = float(np.mean(scores < 400) * 100)
pct_above_800 = float(np.mean(scores > 800) * 100)

print(f'Check 4 - Score Distribution:')
print(f'  Min:  {score_min:.1f}')
print(f'  Max:  {score_max:.1f}')
print(f'  Mean: {score_mean:.1f}')
print(f'  Std:  {score_std:.1f}')
print(f'  % below 400: {pct_below_400:.2f}%')
print(f'  % above 800: {pct_above_800:.2f}%')
score_range_ok = (score_min < 800) and (score_max > 400)
extreme_ok = (pct_below_400 <= 5.0) and (pct_above_800 <= 5.0)
print(f'  Range overlaps [400,800]: {"PASS" if score_range_ok else "FAIL"}')
print(f'  Extremes <= 5%: {"PASS" if extreme_ok else "WARN"}')

In [ ]:
# ---- Check 5: Decile Monotonicity ----
score_df = pd.DataFrame({'score': scores, 'default': y.values})
score_df['decile'] = pd.qcut(score_df['score'], 10, labels=False, duplicates='drop')
decile_stats = score_df.groupby('decile').agg(
    count=('default', 'count'),
    n_defaults=('default', 'sum'),
    avg_score=('score', 'mean')
).reset_index()
decile_stats['default_rate'] = decile_stats['n_defaults'] / decile_stats['count']

# Check monotonicity: higher score = lower risk, so default rate should decrease with decile
dr_values = decile_stats['default_rate'].values
# monotonic decreasing means each next value <= previous
is_monotonic = all(dr_values[i] >= dr_values[i+1] for i in range(len(dr_values)-1))

print('Check 5 - Decile Monotonicity:')
print(decile_stats[['decile', 'count', 'n_defaults', 'avg_score', 'default_rate']].to_string(index=False))
print(f'\nMonotonically decreasing default rate: {"PASS" if is_monotonic else "FAIL"}')

## 6. Cross-Validation and Bootstrap Validation

In [ ]:
# ---- Check 6: Cross-validation stability ----
# Prepare WoE-encoded data for validation
val_db = dev_db[selected_variables + [target]].copy()

# K-fold cross-validation
kfold_res = pdt.kfold_vld(
    model=logit_res,
    db=val_db,
    target=target,
    predictors=selected_variables,
    k=10
)

print('K-Fold Cross-Validation Results:')
print(kfold_res.summary.to_string())

# Extract mean CV AUC
cv_auc_mean = float(kfold_res.summary.loc[kfold_res.summary.index[-1], 'auc']) if 'auc' in kfold_res.summary.columns else float(kfold_res.iter['auc'].mean())
cv_auc_diff = abs(model_auc - cv_auc_mean)
print(f'\nDevelopment AUC: {model_auc:.4f}')
print(f'CV Mean AUC: {cv_auc_mean:.4f}')
print(f'Difference: {cv_auc_diff:.4f}')
cv_stable = cv_auc_diff <= 0.03
print(f'Stability (diff <= 0.03): {"PASS" if cv_stable else "FAIL"}')

In [ ]:
# Bootstrap validation
boots_res = pdt.boots_vld(
    model=logit_res,
    db=val_db,
    target=target,
    predictors=selected_variables,
    B=500
)

print('Bootstrap Validation Results:')
print(boots_res.summary.to_string())

boots_auc_mean = float(boots_res.iter['auc'].mean())
boots_auc_diff = abs(model_auc - boots_auc_mean)
print(f'\nBootstrap Mean AUC: {boots_auc_mean:.4f}')
print(f'Difference from dev: {boots_auc_diff:.4f}')

## 7. Plots

In [ ]:
# ---- ROC Curve ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, color=BLUE, lw=2, label=f'MIV Model (AUC = {model_auc:.4f})')
ax.plot([0, 1], [0, 1], color=GREY, lw=1, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — MIV Stepwise Model')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04a_roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# ---- Score Distribution ----
fig, ax = plt.subplots(figsize=(10, 6))
defaults = scores[y.values == 1]
non_defaults = scores[y.values == 0]

ax.hist(non_defaults, bins=30, alpha=0.6, color=BLUE, label='Non-Default', density=True)
ax.hist(defaults, bins=30, alpha=0.6, color=RED, label='Default', density=True)
ax.axvline(x=score_mean, color=GREY, linestyle='--', label=f'Mean = {score_mean:.0f}')
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution — MIV Model')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04a_score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# ---- Score Decile Table (as figure) ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(decile_stats['decile'], decile_stats['default_rate'], color=RED, alpha=0.7, label='Default Rate')
ax2 = ax.twinx()
ax2.plot(decile_stats['decile'], decile_stats['avg_score'], color=BLUE, marker='o', lw=2, label='Avg Score')
ax.set_xlabel('Score Decile (0=lowest score, 9=highest score)')
ax.set_ylabel('Default Rate', color=RED)
ax2.set_ylabel('Average Score', color=BLUE)
ax.set_title('Score Decile Analysis — MIV Model')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04a_score_decile_table.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# ---- Coefficient Plot ----
coef_df = summary_df[summary_df['variable'] != 'const'].copy()
coef_df = coef_df.sort_values('coef', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [BLUE if c > 0 else RED for c in coef_df['coef']]
ax.barh(coef_df['variable'], coef_df['coef'], color=colors, alpha=0.8)
ax.errorbar(coef_df['coef'], coef_df['variable'],
            xerr=[coef_df['coef'] - coef_df['ci_lower'], coef_df['ci_upper'] - coef_df['coef']],
            fmt='none', ecolor=GREY, capsize=3)
ax.axvline(x=0, color=GREY, linestyle='--', lw=0.8)
ax.set_xlabel('Coefficient')
ax.set_title('Logistic Regression Coefficients — MIV Model')
ax.grid(True, alpha=0.3, axis='x')
plt.savefig(f'{RUN_DIR}/figures/04a_coefficient_plot.png', dpi=150, bbox_inches='tight')
plt.close()

## 8. Write Model Parameters

In [ ]:
# Build model_params_miv.json
model_params = {
    'selection_method': 'miv',
    'selected_variables': selected_variables,
    'woe_mappings': woe_mappings,
    'coefficients': {var: round(float(logit_res.params[var]), 6) for var in selected_variables},
    'intercept': round(float(logit_res.params['const']), 6),
    'score_params': {
        'base_score': 600,
        'base_odds': 50,
        'pdo': 20
    },
    'model_auc': round(float(model_auc), 4),
    'model_gini': round(float(model_gini), 4),
    'model_ks': round(float(model_ks), 4)
}

with open(f'{RUN_DIR}/pipeline/model_params_miv.json', 'w') as f:
    json.dump(model_params, f, indent=2)

print(f'Model parameters written to {RUN_DIR}/pipeline/model_params_miv.json')
print(f'Selected variables: {selected_variables}')
print(f'Coefficients: {model_params["coefficients"]}')
print(f'Intercept: {model_params["intercept"]}')

## Stage Summary

| Item | Value | Status |
|---|---|---|
| Selection method | MIV Stepwise | - |
| Variables selected | See output above | PASS |
| AUC | See output above | See benchmark |
| Gini | See output above | See benchmark |
| KS | See output above | See benchmark |
| Coefficient signs | See check 2 | See check |
| VIF (max) | See check 3 | See check |
| Score range | See check 4 | See check |
| Decile monotonicity | See check 5 | See check |
| CV stability | See check 6 | See check |

**Flags for human review:** See output above

**Recommended action for next stage:** Compare with 04b (XGBoost) and 04c (Forward) models in stage 04x